# Laboratorio #4 - Aprendizaje por Refuerzo
* Paula Barillas - 22764
* Gerardo Pineda - 22880
* Mónica Salvatierra - 22249
* Bianca Calderón - 22272

- Link del repositorio: https://github.com/alee2602/LAB4-RL


## **Task 1**

Una empresa de logística de última milla está evaluando el uso de robots autónomos para la gestión interna de su almacén principal. El almacén se modela como una cuadrícula de 8 × 8con puntos de recogida, puntos de entrega, zonas de penalización por congestión, y obstáculos fijos. El equipo de ingeniería necesita comparar dos estrategias de aprendizaje antes de comprometer recursos en un sistema completo: una política conservadora que aprenda a navegar de forma segura durante el entrenamiento, y una política agresiva que busque la ruta óptima sin importar los riesgos durante la exploración. Su grupo ha sido contratado para implementar ambas estrategias usando SARSA y Q-Learning respectivamente, comparar su comportamiento empírico, y producir un dictamen técnico con recomendaciones concretas para la gerencia

Diseñen formalmente el MDP que representa el almacén. El diseño debe especificar:

**1. El espacio de estados y el espacio de acciones. Justifiquen cada decisión considerando la interfaz de Gymnasium: observation_space y action_space deben ser instancias de gymnasium.spaces.Discrete o gymnasium.spaces.Box según corresponda. Argumenten cuál es más apropiado para este dominio.**

En el presente contexto, lo más apropiado es gymnasium.spaces.Discrete para ambos espacios, no Box. Cada celda de la cuadrícula 8x8 se codifica como un entero único con `estado = fila * 8 + columna`, dando Discrete(64), porque el agente siempre ocupa una posición exacta y no hay noción continua de ubicación. El uso de box tendría sentido con coordenadas continuas o imágenes como observación, pero aquí solo añadiría complejidad innecesaria. Para las acciones, el agente se mueve en cuatro direcciones, lo que se representa con Discrete(4).

**2. La función de recompensa con al menos tres componentes: recompensa por entrega exitosa, penalización por zona de congestión, y penalización por paso. Justifiquen la magnitud relativa de cada componente y argumenten qué comportamiento indeseable produciría una ponderación incorrecta de alguno de ellos.**

- **Recompensa por entrega exitosa:** un valor grande y positivo, por ejemplo +70, otorgado solo al llegar al punto de entrega, para que domine claramente sobre las penalizaciones acumuladas en una trayectoria típica.

- **Penalización por zona de congestión:** un valor negativo moderado, por ejemplo -5, aplicado al estar sobre una celda de congestión, suficiente para desincentivar pero sin que las rutas alternativas sean incoherentes.

- **Penalización por paso:** un valor pequeño y negativo, por ejemplo -1, en cada transición, para incentivar el uso de rutas cortas.

En este caso, una mala calibración produce comportamientos indeseables. Si el paso pesa demasiado frente a la congestión, el agente cruzará zonas de riesgo con tal de ahorrar tiempo. Si la congestión pesa demasiado frente a la entrega, el agente empezará a dar vueltas o evitará completar la tarea. Si la entrega no domina lo suficiente, el agente puede minimizar penalizaciones sin nunca llegar a la meta.

**3. La condición de terminación del episodio: ¿cuándo termina un episodio? ¿Es apropiado tener un
límite máximo de pasos? Justifiquen.**

- El episodio termina cuando el agente llega al punto de entrega. También conviene agregar un límite máximo de pasos, por ejemplo 100, como truncamiento. Esto evita que una política casi aleatoria en las primeras fases pueda quedar atrapada en ciclos infinitos. En Gymnasium esto se distingue con `terminated` (llegó a la meta) y `truncated` (se acabó el tiempo), siendo de utilidad para posteriores análisis. 

**4. El diseño del mapa 8 × 8: ubiquen al menos dos zonas de congestión adyacentes a rutas de altarecompensa. Esta configuración específica es crítica para que la comparación entre SARSA y Q-Learning sea informativa. Expliquen por qué esa configuración espacial genera el comportamiento
diferencial esperado entre ambos algoritmos.**

Consideramos que conviene ubicar al menos dos zonas de congestión sobre o muy cerca del camino más corto entre recogida y entrega. Por ejemplo, con inicio en la esquina superior izquierda y entrega en la inferior derecha, una franja de congestión puede cruzar la diagonal natural, dejando una ruta alternativa más larga que la rodea.

Esta configuración es la que hace útil la comparación entre algoritmos. Q-Learning es off-policy y aprende el valor máximo del siguiente estado sin importar la acción exploratoria tomada, por lo que su política greedy final puede explotar el camino corto arriesgado si la recompensa neta lo justifica. SARSA es on-policy y actualiza según la acción que realmente tomará bajo su política, incluyendo exploración epsilon-greedy, lo que lo hace más sensible al riesgo de caer en congestión por una acción aleatoria y lo lleva a preferir rutas más conservadoras. Sin esta adyacencia entre congestión y ruta óptima, ambos algoritmos convergerían a políticas casi idénticas, por lo que no habría punto de comparación.

## **Task 2**

**1. Para el entorno que diseñaron, predigan formalmente cuál algoritmo, SARSA o Q-Learning, producirá
mayor recompensa acumulada durante el entrenamiento y cuál producirá mayor recompensa
durante la evaluación con política greedy pura. Justifiquen cada predicción usando las propiedades
on-policy y off-policy de cada algoritmo y la estructura específica de su mapa**

Q-Learning (off-policy) actualiza con $max_{a'} Q(s',a')$, ignorando que la exploración ε-greedy real puede empujar al agente hacia la congestión. Esto le hace sobrevalorar el camino corto y arriesgado. SARSA (on-policy) actualiza con Q(s',a') donde a' sí viene de la política ε-greedy real, así que "siente" el riesgo de caer en congestión por una acción exploratoria y aprende a preferir la ruta larga y segura.

En entrenamiento, ambos exploran con ε>0: Q-Learning cae más veces en congestión porque su política greedy está pegada al borde y SARSA acumula más recompensa durante el entrenamiento. En evaluación greedy ε=0: Q-Learning ya convergió a Q* (el óptimo real, sin castigo por exploración) y toma el atajo sin pagar el costo. SARSA quedó sesgado hacia la ruta conservadora que aprendió, aunque ya no sea necesaria. Por lo tanto Q-Learning gana en evaluación. En general, SARSA gana en entrenamiento y Q-Learning gana en evaluación greedy pura.


**2. Argumenten cómo afecta el valor de εal comportamiento diferencial entre SARSA y Q-Learning en su
entorno. ¿Existe un valor de εpara el cual ambos algoritmos convergen a políticas idénticas?
Justifiquen matemáticamente**

El parámetro ε controla la magnitud de la brecha entre ambos algoritmos, porque es precisamente el término que aparece en el target de SARSA y no en el de Q-Learning. Como $Σ_{a'} π_ε(a'|s')·Q(s',a') = (1−ε)·max_{a'}Q(s',a') + (ε/|A|)·Σ_{a'}Q(s',a')$, la diferencia entre ambos targets es proporcional a ε: mientras mayor sea ε, mayor peso reciben las acciones subóptimas (incluyendo caer en congestión), y mayor será el sesgo conservador de SARSA respecto a Q-Learning. Con ε bajo, ambos targets se parecen más y la brecha entre las políticas se reduce.

Sí, en el límite ε→0. Cuando ε->0, $π_ε(a'|s') -> 𝟙[a'=argmax]$, por lo que: $Σ_{a'} π_ε(a'|s')·Q(s',a') -> max_{a'} Q(s',a')$. Las dos ecuaciones de punto fijo coinciden, dando Q_SARSA -> Q*. Esto es consistente con la teoría de convergencia de Greedy in the Limit with Infinite Exploration, si ε se decae a 0 con una tasa apropiada durante el entrenamiento, manteniendo exploración infinita en el límite temprano, tanto SARSA como Q-Learning convergen a la misma política óptima determinística. Cabe notar que con ε estrictamente igual a 0 desde el inicio no habría exploración y el aprendizaje podría quedar atrapado en óptimos locales sin garantía de cobertura completa del espacio estado-acción; por eso la convergencia formal se enuncia como un límite: $ε_t → 0$, no como un valor fijo utilizable en la práctica.


**3. Para su función de recompensa específica, calculen una cota superior del valor óptimo del
estado inicial, asumiendo que el agente siempre toma la ruta más corta sin pasar por zonas de
congestión. Usen esta cota como referencia para evaluar qué tan cerca llegan sus implementaciones
al óptimo teórico**

Teniendo en cuenta que grid 8×8 con inicio en (0,0) y entrega en (7,7), recompensa de entrega +70, penalización por paso −1, γ como factor de descuento, tomamos γ=1 por ser un entorno episódico de horizonte finito, típico en estas tareas. la distancia Manhattan entre (0,0) y (7,7) es d = |7−0| + |7−0| = 14 pasos, lograble moviéndose solo en dirección derecha/abajo, sin desperdiciar ningún paso. Esta es la ruta más corta posible en la cuadrícula, independientemente de si cruza o no congestión.

Con Cota superior γ=1: $V*(s₀) ≤ 70 + 14·(−1) = 70 − 14 = 56$ y con γ descontando cada paso: $V*(s₀) ≤ −Σ_{t=0}^{d−1} γ^t + 70·γ^d = −(1−γ^d)/(1−γ) + 70·γ^d$

Esta cifra asume la mejor situación posible, el camino más corto matemáticamente permitido por la geometría de la cuadrícula, sin ningún paso adicional y sin pisar ninguna celda de congestión. Dado que en nuestro diseño colocamos deliberadamente zonas de congestión sobre o cerca de la diagonal natural entre inicio y entrega, es muy probable que la ruta realmente óptima, la que maximiza recompensa esperada considerando la penalización de congestión, deba desviarse de esa diagonal, incrementando su longitud real a d_real > 14. Por lo tanto, el valor óptimo verdadero cumplirá: $V*(s₀) ≤ 56$. Esta cota sirve como referencia, si al evaluar las políticas entrenadas obtenemos recompensas cercanas a 56, sabremos que el agente encontró o casi el atajo óptimo ignorando el riesgo de congestión, en cambio si obtenemos valores notablemente menores, eso refleja el costo que cada algoritmo paga por evitar o tomar el riesgo de la zona de congestión.
